# NAS-YOLO: Noise-Aware State Modeling for Robust Real-Time Object Detection
## ECCV 2026 — Full Experiment Pipeline

**One-click reproducible experiments**: GPU auto-detection, optimal batch sizing, full training + evaluation.

| Stage | Description | Time (A100) | Time (T4) |
|-------|-------------|-------------|------------|
| 0 | Setup & Smoke Test | ~5 min | ~5 min |
| 1 | Training (nano/small/medium) | ~18h | ~72h |
| 2 | Ablation (3 variants) | ~6h | ~24h |
| 3 | Evaluation (mAP + mAP-C + BSI) | ~2h | ~8h |
| 4 | Tables & Figures | <1 min | <1 min |

In [ ]:
#@title **Stage 0: Environment Setup** { display-mode: "form" }
#@markdown Installs dependencies, detects GPU, sets optimal hyperparameters.

import subprocess, os, sys, json, time

# === Install Dependencies ===
print("[1/4] Installing dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu121"],
    check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "numpy", "pandas", "matplotlib", "seaborn", "pycocotools",
    "pyyaml", "tqdm", "Pillow", "scipy"],
    check=True, capture_output=True)
print("  Dependencies installed.")

# === Clone Repository ===
print("[2/4] Setting up repository...")
REPO_DIR = "/content/Study-with-AI-Finance-Coach"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/DrJinHoChoi/Study-with-AI-Finance-Coach.git", REPO_DIR],
                   check=True, capture_output=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f"  Working directory: {os.getcwd()}")

# === GPU Detection & Optimal Config ===
print("[3/4] Detecting GPU...")
import torch

assert torch.cuda.is_available(), "CUDA not available! Switch to GPU runtime."

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = torch.cuda.get_device_properties(0).total_mem / 1e9
NUM_GPUS = torch.cuda.device_count()

# Optimal settings per GPU type
GPU_CONFIGS = {
    "A100": {"batch_nano": 64, "batch_small": 32, "batch_medium": 16, "workers": 8, "tier": "high"},
    "A10G": {"batch_nano": 48, "batch_small": 24, "batch_medium": 12, "workers": 8, "tier": "high"},
    "V100": {"batch_nano": 32, "batch_small": 16, "batch_medium": 8,  "workers": 8, "tier": "mid"},
    "T4":   {"batch_nano": 16, "batch_small": 8,  "batch_medium": 4,  "workers": 4, "tier": "low"},
    "L4":   {"batch_nano": 32, "batch_small": 16, "batch_medium": 8,  "workers": 6, "tier": "mid"},
}

# Match GPU
gpu_cfg = GPU_CONFIGS["T4"]  # default fallback
for key in GPU_CONFIGS:
    if key in GPU_NAME:
        gpu_cfg = GPU_CONFIGS[key]
        break

# Auto-adjust for multi-GPU
if NUM_GPUS > 1:
    for k in ["batch_nano", "batch_small", "batch_medium"]:
        gpu_cfg[k] *= NUM_GPUS

print(f"  GPU: {GPU_NAME} ({GPU_MEM_GB:.1f} GB) x {NUM_GPUS}")
print(f"  PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
print(f"  Tier: {gpu_cfg['tier']} | Batch sizes: nano={gpu_cfg['batch_nano']}, small={gpu_cfg['batch_small']}, medium={gpu_cfg['batch_medium']}")

# === Validate Codebase ===
print("[4/4] Validating codebase...")
result = subprocess.run([sys.executable, "nas_yolo/validate_code.py"], capture_output=True, text=True)
if "All checks passed" in result.stdout:
    lines = [l for l in result.stdout.split("\n") if "Passed:" in l or "Failed:" in l]
    print(f"  {' | '.join(l.strip() for l in lines)}")
else:
    print(result.stdout[-500:])
    raise RuntimeError("Code validation failed!")

# Save config for later cells
ENV = {
    "gpu_name": GPU_NAME, "gpu_mem_gb": GPU_MEM_GB, "num_gpus": NUM_GPUS,
    **gpu_cfg, "repo_dir": REPO_DIR,
    "data_dir": os.path.join(REPO_DIR, "data/coco"),
    "runs_dir": os.path.join(REPO_DIR, "runs"),
}
os.makedirs(ENV["runs_dir"], exist_ok=True)
print(f"\n{'='*60}")
print(f"Environment ready. GPU tier: {gpu_cfg['tier']}")
print(f"{'='*60}")

In [ ]:
#@title **Smoke Test: Model Construction + Forward + Training Step** { display-mode: "form" }

import torch
import time
import sys
sys.path.insert(0, '.')

from nas_yolo.models.nas_yolo import NASYOLO, NASYOLOWithTCL, MODEL_CONFIGS
from nas_yolo.models.noise_gate import SpectralNoiseEstimator
from nas_yolo.models.temporal_buffer import PseudoTemporalGenerator

device = torch.device("cuda")

# --- Model Construction ---
print("=" * 60)
print("1. Model Construction")
print("=" * 60)
for scale in ["nano", "small", "medium"]:
    model = NASYOLO(num_classes=80, model_scale=scale).to(device).eval()
    info = model.get_model_info()
    x = torch.randn(1, 3, 640, 640, device=device)
    with torch.no_grad():
        outputs = model(x)
    total_preds = sum(o.shape[2] * o.shape[3] for o in outputs["cls_preds"])
    print(f"  NAS-YOLO-{scale[0]}: {info['total_params_M']:.2f}M params, "
          f"NAS overhead: {info['nas_overhead_pct']:.1f}%, "
          f"Predictions: {total_preds}")
    del model; torch.cuda.empty_cache()

# --- Temporal Forward ---
print(f"\n{'='*60}")
print("2. Temporal State Propagation")
print("=" * 60)
model = NASYOLO(num_classes=80, model_scale="nano").to(device).eval()
sequence = [torch.randn(1, 3, 640, 640, device=device) for _ in range(5)]
with torch.no_grad():
    outputs_seq = model.forward_temporal(sequence)
print(f"  Temporal sequence: {len(outputs_seq)} frames")
print(f"  State initialized: {model.temporal_buffer.is_initialized}")

model.reset_temporal_state()
out1 = model(sequence[0])
model.reset_temporal_state()
_ = model(sequence[0])  # prime
out2 = model(sequence[0])
diff = sum((a - b).abs().mean().item() for a, b in zip(out1["obj_preds"], out2["obj_preds"]))
print(f"  State effect (obj diff): {diff:.6f}")
del model; torch.cuda.empty_cache()

# --- Noise Estimation ---
print(f"\n{'='*60}")
print("3. Noise Estimation")
print("=" * 60)
estimator = SpectralNoiseEstimator(in_channels=3, noise_dim=32).to(device).eval()
clean = torch.rand(1, 3, 640, 640, device=device)
noisy = (clean + torch.randn_like(clean) * 0.3).clamp(0, 1)
with torch.no_grad():
    d_clean = estimator(clean)
    d_noisy = estimator(noisy)
print(f"  Clean norm: {d_clean.norm():.4f}, Noisy norm: {d_noisy.norm():.4f}")
print(f"  Diff: {(d_clean - d_noisy).abs().mean():.4f}")
del estimator; torch.cuda.empty_cache()

# --- 1-Step Training ---
print(f"\n{'='*60}")
print("4. Training Step (1 iteration)")
print("=" * 60)
model = NASYOLOWithTCL(num_classes=80, model_scale="nano", tcl_weight=0.5).to(device).train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
x = torch.randn(2, 3, 320, 320, device=device)
targets = [
    {"boxes": torch.tensor([[50., 50., 150., 150.]], device=device), "labels": torch.tensor([3], device=device)},
    {"boxes": torch.tensor([[20., 20., 100., 100.]], device=device), "labels": torch.tensor([1], device=device)},
]
gen = PseudoTemporalGenerator(sequence_length=3)
seq = gen(x)
targets_seq = [targets] * len(seq)
outputs_seq, tcl_loss = model.forward_temporal_with_tcl(seq, targets_seq)
det_loss = sum(o["losses"]["loss"] for o in outputs_seq if "losses" in o) / len(outputs_seq)
loss = det_loss + tcl_loss
optimizer.zero_grad()
loss.backward()
optimizer.step()
print(f"  Total loss: {loss.item():.4f} (det: {det_loss.item():.4f}, tcl: {tcl_loss.item():.4f})")
del model; torch.cuda.empty_cache()

# --- Latency Benchmark ---
print(f"\n{'='*60}")
print("5. Latency Benchmark")
print("=" * 60)
for scale in ["nano", "small", "medium"]:
    model = NASYOLO(num_classes=80, model_scale=scale).to(device).eval()
    x = torch.randn(1, 3, 640, 640, device=device)
    for _ in range(20): model(x)  # warmup
    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(100):
        with torch.no_grad(): model(x)
    torch.cuda.synchronize()
    ms = (time.time() - t0) / 100 * 1000
    print(f"  NAS-YOLO-{scale[0]}: {ms:.1f}ms ({1000/ms:.0f} FPS)")
    del model; torch.cuda.empty_cache()

# --- BSI Metric ---
print(f"\n{'='*60}")
print("6. BSI Metric")
print("=" * 60)
import numpy as np
from nas_yolo.metrics.box_stability import BoxStabilityIndex, FrameDetections
bsi = BoxStabilityIndex()
boxes = np.array([[100, 100, 200, 200]], dtype=np.float32)
perfect = [FrameDetections(boxes=boxes.copy(), scores=np.array([0.9]),
                           class_ids=np.array([0]), frame_idx=t) for t in range(10)]
r = bsi.compute(perfect)
print(f"  Perfect BSI: {r.bsi:.4f}")
jittery = [FrameDetections(boxes=boxes + np.random.randn(1,4).astype(np.float32)*5,
                           scores=np.array([0.9]), class_ids=np.array([0]), frame_idx=t) for t in range(10)]
r2 = bsi.compute(jittery)
print(f"  Jittery BSI: {r2.bsi:.4f}")

print(f"\n{'='*60}")
print("All smoke tests passed!")
print(f"{'='*60}")

In [ ]:
#@title **Download COCO 2017** { display-mode: "form" }
#@markdown Downloads COCO 2017 train/val + annotations (~25 GB total).

import os
DATA_DIR = ENV["data_dir"]
os.makedirs(DATA_DIR, exist_ok=True)

def download_if_missing(url, dest_dir, folder_name):
    target = os.path.join(dest_dir, folder_name)
    if os.path.exists(target):
        print(f"  {folder_name} already exists, skipping.")
        return
    zip_name = url.split("/")[-1]
    zip_path = os.path.join(dest_dir, zip_name)
    print(f"  Downloading {zip_name}...")
    !wget -q --show-progress -O {zip_path} {url}
    print(f"  Extracting {zip_name}...")
    !unzip -q {zip_path} -d {dest_dir} && rm {zip_path}
    print(f"  {folder_name} ready.")

download_if_missing("http://images.cocodataset.org/zips/train2017.zip", DATA_DIR, "train2017")
download_if_missing("http://images.cocodataset.org/zips/val2017.zip", DATA_DIR, "val2017")
download_if_missing("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", DATA_DIR, "annotations")

# Verify
train_count = len(os.listdir(os.path.join(DATA_DIR, "train2017")))
val_count = len(os.listdir(os.path.join(DATA_DIR, "val2017")))
print(f"\nCOCO 2017: {train_count} train images, {val_count} val images")
assert train_count > 100000, f"Expected >100k train images, got {train_count}"
assert val_count > 4000, f"Expected >4k val images, got {val_count}"
print("COCO dataset verified!")

---
## Stage 1: Training
Trains NAS-YOLO nano, small, and medium variants on COCO 2017 (300 epochs each).

In [ ]:
#@title **Train NAS-YOLO (all scales)** { display-mode: "form" }
#@markdown Trains nano → small → medium sequentially. Skips if checkpoint exists.

import subprocess, os, time

SCALES = [
    ("nano",   "nas_yolo/configs/nas_yolo_n.yaml", ENV["batch_nano"]),
    ("small",  "nas_yolo/configs/default.yaml",     ENV["batch_small"]),
    ("medium", "nas_yolo/configs/nas_yolo_m.yaml",  ENV["batch_medium"]),
]

for scale, config, batch_size in SCALES:
    run_dir = os.path.join(ENV["runs_dir"], f"nas_yolo_{scale}")
    
    if os.path.exists(os.path.join(run_dir, "best.pt")):
        print(f"[SKIP] NAS-YOLO-{scale[0]} (checkpoint exists at {run_dir}/best.pt)")
        continue
    
    print(f"\n{'='*60}")
    print(f"Training NAS-YOLO-{scale[0]} | batch={batch_size} | config={config}")
    print(f"{'='*60}")
    
    os.makedirs(run_dir, exist_ok=True)
    t0 = time.time()
    
    cmd = [
        "python", "-m", "nas_yolo.scripts.train",
        "--config", config,
        "--data-root", ENV["data_dir"],
        "--output-dir", run_dir,
        "--batch-size", str(batch_size),
        "--workers", str(ENV["workers"]),
        "--amp",
    ]
    
    # Use torchrun for multi-GPU
    if ENV["num_gpus"] > 1:
        cmd = ["torchrun", f"--nproc_per_node={ENV['num_gpus']}"] + cmd[1:]
    
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    
    elapsed = time.time() - t0
    print(f"\nNAS-YOLO-{scale[0]} training: {elapsed/3600:.1f}h")
    
    if proc.returncode != 0:
        print(f"WARNING: Training exited with code {proc.returncode}")

print("\nAll training complete!")

---
## Stage 2: Ablation Study
Three ablation variants (all using nano scale for efficiency):
1. **w/o Temporal**: No SSM temporal state
2. **w/o Noise Gate**: No noise-aware gating
3. **w/o TCL**: No temporal consistency loss

In [ ]:
#@title **Run Ablation Experiments** { display-mode: "form" }

ABLATIONS = [
    ("no_temporal",   "nas_yolo/configs/ablation/no_temporal.yaml"),
    ("no_noise_gate", "nas_yolo/configs/ablation/no_noise_gate.yaml"),
    ("no_tcl",        "nas_yolo/configs/ablation/no_tcl.yaml"),
]

for name, config in ABLATIONS:
    run_dir = os.path.join(ENV["runs_dir"], f"ablation_{name}")
    
    if os.path.exists(os.path.join(run_dir, "best.pt")):
        print(f"[SKIP] Ablation {name} (checkpoint exists)")
        continue
    
    print(f"\n{'='*60}")
    print(f"Ablation: {name}")
    print(f"{'='*60}")
    
    os.makedirs(run_dir, exist_ok=True)
    t0 = time.time()
    
    cmd = [
        "python", "-m", "nas_yolo.scripts.train",
        "--config", config,
        "--data-root", ENV["data_dir"],
        "--output-dir", run_dir,
        "--batch-size", str(ENV["batch_nano"]),
        "--workers", str(ENV["workers"]),
        "--amp",
    ]
    
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    
    elapsed = time.time() - t0
    print(f"\nAblation {name}: {elapsed/3600:.1f}h")

print("\nAll ablation experiments complete!")

---
## Stage 3: Comprehensive Evaluation
For each trained model:
1. **Clean mAP** (COCO val2017)
2. **mAP-C** (15 corruptions x 5 severities)
3. **BSI** (Box Stability Index on pseudo-temporal sequences)
4. **Latency** (FPS benchmark)

In [ ]:
#@title **Evaluate All Models** { display-mode: "form" }

import json

# Models to evaluate
EVAL_MODELS = [
    # (name, checkpoint, scale)
    ("nas_yolo_nano",   f"{ENV['runs_dir']}/nas_yolo_nano/best.pt",   "nano"),
    ("nas_yolo_small",  f"{ENV['runs_dir']}/nas_yolo_small/best.pt",  "small"),
    ("nas_yolo_medium", f"{ENV['runs_dir']}/nas_yolo_medium/best.pt", "medium"),
    # Ablations
    ("ablation_no_temporal",   f"{ENV['runs_dir']}/ablation_no_temporal/best.pt",   "nano"),
    ("ablation_no_noise_gate", f"{ENV['runs_dir']}/ablation_no_noise_gate/best.pt", "nano"),
    ("ablation_no_tcl",        f"{ENV['runs_dir']}/ablation_no_tcl/best.pt",        "nano"),
]

all_results = {}

for name, ckpt, scale in EVAL_MODELS:
    if not os.path.exists(ckpt):
        print(f"[SKIP] {name} (no checkpoint at {ckpt})")
        continue
    
    eval_dir = os.path.join(ENV["runs_dir"], f"eval_{name}")
    os.makedirs(eval_dir, exist_ok=True)
    output_file = os.path.join(eval_dir, "eval_results.json")
    
    if os.path.exists(output_file):
        print(f"[CACHED] {name}")
        with open(output_file) as f:
            all_results[name] = json.load(f)
        continue
    
    print(f"\n{'='*60}")
    print(f"Evaluating: {name} (scale={scale})")
    print(f"{'='*60}")
    
    cmd = [
        "python", "-m", "nas_yolo.scripts.evaluate",
        "--checkpoint", ckpt,
        "--model-scale", scale,
        "--data-root", ENV["data_dir"],
        "--output-dir", eval_dir,
        "--batch-size", str(ENV["batch_small"]),
        "--full",
    ]
    
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    
    if os.path.exists(output_file):
        with open(output_file) as f:
            all_results[name] = json.load(f)

# Save combined results
combined_path = os.path.join(ENV["runs_dir"], "all_eval_results.json")
with open(combined_path, "w") as f:
    json.dump(all_results, f, indent=2, default=str)
print(f"\nAll results saved to {combined_path}")

In [ ]:
#@title **Latency Benchmark (All Scales)** { display-mode: "form" }

!python -m nas_yolo.scripts.benchmark --all-scales --overhead-analysis \
    --output {ENV['runs_dir']}/benchmark_results.json

---
## Stage 4: Generate Paper Tables & Figures

In [ ]:
#@title **Generate LaTeX Tables** { display-mode: "form" }

!python nas_yolo/experiments/generate_tables.py \
    --results-dir {ENV['runs_dir']} \
    --output-dir {ENV['runs_dir']}/tables

# Display generated tables
import glob
for tex_file in sorted(glob.glob(f"{ENV['runs_dir']}/tables/*.tex")):
    print(f"\n{'='*60}")
    print(f"File: {os.path.basename(tex_file)}")
    print(f"{'='*60}")
    with open(tex_file) as f:
        print(f.read())

In [ ]:
#@title **Visualize Results** { display-mode: "form" }

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 12, "figure.dpi": 150})

# 1. Training curves (if training logs exist)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for scale, color in [("nano", "#2196F3"), ("small", "#4CAF50"), ("medium", "#FF9800")]:
    log_file = os.path.join(ENV["runs_dir"], f"nas_yolo_{scale}", "train.log")
    if not os.path.exists(log_file):
        continue
    losses = []
    with open(log_file) as f:
        for line in f:
            if "Loss:" in line:
                try:
                    loss_val = float(line.split("Loss:")[1].split("|")[0].strip())
                    losses.append(loss_val)
                except (ValueError, IndexError):
                    pass
    if losses:
        axes[0].plot(losses, label=f"NAS-YOLO-{scale[0]}", color=color, linewidth=1.5)

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. mAP comparison
if all_results:
    models = []
    clean_maps = []
    map_cs = []
    bsis = []
    for name, res in all_results.items():
        models.append(name.replace("nas_yolo_", "").replace("ablation_", "abl:"))
        clean_maps.append(res.get("clean", {}).get("mAP50", 0) * 100)
        map_cs.append(res.get("corruption", {}).get("mAP_C", 0) * 100)
        bsis.append(res.get("stability", {}).get("BSI", 0))
    
    x = range(len(models))
    axes[1].bar([i - 0.15 for i in x], clean_maps, 0.3, label="Clean mAP@50", color="#2196F3")
    axes[1].bar([i + 0.15 for i in x], map_cs, 0.3, label="mAP-C", color="#FF5722")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(models, rotation=30, ha="right", fontsize=9)
    axes[1].set_ylabel("mAP (%)")
    axes[1].set_title("Detection Accuracy")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis="y")
    
    axes[2].bar(x, bsis, color="#4CAF50")
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(models, rotation=30, ha="right", fontsize=9)
    axes[2].set_ylabel("BSI")
    axes[2].set_title("Box Stability Index")
    axes[2].set_ylim(0, 1)
    axes[2].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
fig_path = os.path.join(ENV["runs_dir"], "experiment_summary.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved to {fig_path}")

In [ ]:
#@title **Results Summary** { display-mode: "form" }

print(f"{'='*80}")
print(f"{'Model':<25} {'Params':>8} {'FPS':>6} {'mAP@50':>8} {'mAP-C':>8} {'BSI':>8}")
print(f"{'-'*80}")

for name, res in all_results.items():
    params = res.get("model_info", {}).get("total_params_M", 0)
    fps = res.get("latency", {}).get("throughput_fps", 0)
    clean = res.get("clean", {}).get("mAP50", 0) * 100
    map_c = res.get("corruption", {}).get("mAP_C", 0) * 100
    bsi_val = res.get("stability", {}).get("BSI", 0)
    
    print(f"{name:<25} {params:>7.1f}M {fps:>5.0f} {clean:>7.1f}% {map_c:>7.1f}% {bsi_val:>7.3f}")

print(f"{'='*80}")
print(f"\nGPU: {ENV['gpu_name']}")
print(f"All results: {ENV['runs_dir']}/all_eval_results.json")

In [ ]:
#@title **Save Checkpoints to Google Drive** { display-mode: "form" }
#@markdown Optional: mount Drive and copy results for persistence.

SAVE_TO_DRIVE = False  #@param {type:"boolean"}

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    
    drive_dest = "/content/drive/MyDrive/NAS_YOLO_ECCV2026"
    os.makedirs(drive_dest, exist_ok=True)
    
    !cp -r {ENV['runs_dir']}/*.json {drive_dest}/
    
    # Copy best checkpoints (not all epochs)
    for model_dir in glob.glob(f"{ENV['runs_dir']}/nas_yolo_*") + glob.glob(f"{ENV['runs_dir']}/ablation_*"):
        best_pt = os.path.join(model_dir, "best.pt")
        if os.path.exists(best_pt):
            dest_name = os.path.basename(model_dir)
            os.makedirs(os.path.join(drive_dest, dest_name), exist_ok=True)
            !cp {best_pt} {drive_dest}/{dest_name}/best.pt
            print(f"  Saved {dest_name}/best.pt")
    
    # Copy evaluation results
    !cp -r {ENV['runs_dir']}/eval_* {drive_dest}/ 2>/dev/null || true
    !cp -r {ENV['runs_dir']}/tables {drive_dest}/ 2>/dev/null || true
    
    print(f"\nAll results saved to Google Drive: {drive_dest}")
else:
    print("Skipping Drive save. Set SAVE_TO_DRIVE = True to enable.")